<a href="https://colab.research.google.com/github/dodi-ctrl/PhishingDetector/blob/main/Metadata_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metadata Agent — Multi-Corpus Training

Trains the Random Forest metadata agent on **real RFC 2822 headers** from three sources:

| Source | Label | Provides |
|---|---|---|
| `phishing_pot` (GitHub) | phishing (1) | full headers, modern 2022–2024 captures |
| Nazario `phishing3.mbox` (filtered ≥2022) | phishing (1) | full headers from honeypot |
| Enron ham via HuggingFace | legitimate (0) | body + subject; minimal envelope reconstructed |

This replaces the prior MeAJOR-only approach where SPF/DKIM/DMARC/Received features were all zero.

In [ ]:
# Install dependencies
!pip install -q datasets scikit-learn pandas numpy matplotlib seaborn joblib
print("Installation OK")

In [ ]:
# Clone the repo to get feature_extraction.py, dataset_handling.py, metadata_agent.py
import os

if not os.path.exists('PhishingDetector'):
    !git clone https://github.com/dodi-ctrl/PhishingDetector.git

os.chdir('PhishingDetector')
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))

## Download phishing corpora

- **phishing_pot**: cloned from GitHub — thousands of recent .eml files in `phishing_pot/email/`.
- **Nazario phishing3.mbox**: ~3,000 phishing messages collected by José Nazario.
- **Enron ham** is fetched on demand from HuggingFace inside `build_eml_corpus`.

In [ ]:
# Download phishing_pot (.eml directory) and Nazario phishing3.mbox
import os

if not os.path.exists('phishing_pot'):
    !git clone --depth 1 https://github.com/rf-peixoto/phishing_pot.git

if not os.path.exists('phishing3.mbox'):
    !wget -q https://monkey.org/~jose/phishing/phishing3.mbox

print('phishing_pot exists:', os.path.exists('phishing_pot'))
print('phishing3.mbox exists:', os.path.exists('phishing3.mbox'))
if os.path.exists('phishing_pot'):
    for sub in ('email', 'emails'):
        p = os.path.join('phishing_pot', sub)
        if os.path.isdir(p):
            n = len([f for f in os.listdir(p) if f.endswith('.eml')])
            print(f'  {p}: {n} .eml files')

In [ ]:
# Build the unified .eml corpus
from dataset_handling import build_eml_corpus, extract_metadata_features_from_eml_corpus
from feature_extraction import FeatureExtractor

# phishing_pot stores emails under phishing_pot/email/ (singular).
# Pass both candidate paths so the loader picks whichever exists.
PHISHING_DIRS = ['phishing_pot/email', 'phishing_pot/emails']
PHISHING_MBOX = ['phishing3.mbox']
ENRON_MAX     = 5000

eml_df = build_eml_corpus(
    phishing_dirs=PHISHING_DIRS,
    phishing_mbox_paths=PHISHING_MBOX,
    enron_max=ENRON_MAX,
    nazario_after_year=2022,
)

In [ ]:
# Extract metadata features from real headers
extractor = FeatureExtractor()
features_df, labels = extract_metadata_features_from_eml_corpus(eml_df, extractor)

print('\nFeature matrix shape:', features_df.shape)
print('First 5 features:', list(features_df.columns[:5]))

In [ ]:
# Train / test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_df, labels,
    test_size=0.3,
    random_state=42,
    stratify=labels,
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"  - Legitimate:   {(y_test == 0).sum()}")
print(f"  - Phishing:     {(y_test == 1).sum()}")

In [ ]:
# Train the Metadata Agent
from metadata_agent import MetadataAgent

agent = MetadataAgent(n_estimators=100, max_depth=20, random_state=42)
agent.train(X_train, y_train, validate=True, tune_hyperparameters=False)

In [ ]:
# Evaluate — accuracy, precision, recall, F1, ROC-AUC, FPR/FNR/TNR,
# confusion matrix, classification report, and plots
results = agent.evaluate(X_test, y_test, plot_results=True)

In [ ]:
# Save model and results
agent.save_model('metadata_agent.pkl')
agent.export_results(results, 'metadata_agent_results.json')

print('\n' + '=' * 70)
print('Training complete!')
print(f"  Accuracy:  {results['accuracy']:.4f}")
print(f"  F1-Score:  {results['f1_score']:.4f}")
print(f"  ROC-AUC:   {results['roc_auc']:.4f}")
print('=' * 70)